In [0]:
# Load Tables

from pyspark.sql.functions import col, to_date

fact_df = spark.table("medical_project.gold.fact_encounters")
date_df = spark.table("medical_project.gold.dim_date")

display(fact_df)


In [0]:
# Join with Date Dimension

fact_with_date = fact_df.withColumn(
    "encounter_date",
    to_date(col("start"))
).join(
    date_df,
    col("encounter_date") == col("date"),
    "left"
)

In [0]:
# Aggregate Metrics

from pyspark.sql.functions import count, sum, when

agg_df = fact_with_date.groupBy(
    "year", "month", "payer_id"
).agg(
    count("encounter_id").alias("total_encounters"),
    
    sum(
        when(col("has_payer_coverage") == 0, 1).otherwise(0)
    ).alias("zero_coverage_count")
).orderBy("year", "month")

display(agg_df)

In [0]:
#: Calculate Percentages

agg_df = agg_df.withColumn(
    "zero_coverage_pct",
    (col("zero_coverage_count") / col("total_encounters")) * 100
).withColumn(
    "coverage_pct",
    100 - col("zero_coverage_pct")
)

In [0]:
# Final Output

kpi3 = agg_df.select(
    "year",
    "month",
    "payer_id",
    "total_encounters",
    "zero_coverage_count",
    "zero_coverage_pct",
    "coverage_pct"
).orderBy("year", "month")

display(kpi3)

In [0]:
# Save Table

kpi3.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.kpi_zero_payer_coverage")